# 05 — Accessibility

Computes transit accessibility and walkability features per census tract from OSM.

**Data source:** Overpass API + OSMnx — fully portable.

**Output columns:** `tract_id`, `dist_subway_mean`, `dist_bus_mean`, `transit_stop_count`, `intersection_density`

**Output file:** `csv/05_accessibility.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 800   # meters for transit search around each tract centroid

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

In [ ]:
# ── Overpass helpers ──────────────────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

In [ ]:
# ── Query transit stops per tract ─────────────────────

records = []
n_tracts = len(df_tracts)

for i, row in df_tracts.iterrows():
    tract_id = row["tract_id"]
    lat, lon = row["tract_lat"], row["tract_lon"]
    
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{n_tracts}] Tract {tract_id}")
    
    # Query subway entrances
    subway_query = (f'[out:json][timeout:30];\n'
                    f'(node["railway"="subway_entrance"](around:{QUERY_RADIUS},{lat},{lon});\n'
                    f' node["station"="subway"](around:{QUERY_RADIUS},{lat},{lon}););\n'
                    f'out;')
    
    # Query bus stops
    bus_query = (f'[out:json][timeout:30];\n'
                f'(node["highway"="bus_stop"](around:{QUERY_RADIUS},{lat},{lon});\n'
                f' node["public_transport"="platform"]["bus"="yes"](around:{QUERY_RADIUS},{lat},{lon}););\n'
                f'out;')
    
    try:
        subway_data = query_overpass_cached(subway_query)
        bus_data = query_overpass_cached(bus_query)
    except Exception as e:
        print(f"  ERROR tract {tract_id}: {e}")
        records.append({"tract_id": tract_id, "dist_subway_mean": np.nan,
                        "dist_bus_mean": np.nan, "transit_stop_count": 0})
        continue
    
    # Compute distances to subway
    subway_dists = []
    for el in subway_data.get("elements", []):
        d = haversine(lat, lon, el["lat"], el["lon"])
        subway_dists.append(d)
    
    # Compute distances to bus stops
    bus_dists = []
    for el in bus_data.get("elements", []):
        d = haversine(lat, lon, el["lat"], el["lon"])
        bus_dists.append(d)
    
    dist_subway_mean = round(np.mean(subway_dists), 1) if subway_dists else np.nan
    dist_bus_mean = round(np.mean(bus_dists), 1) if bus_dists else np.nan
    transit_stop_count = len(subway_dists) + len(bus_dists)
    
    records.append({
        "tract_id": tract_id,
        "dist_subway_mean": dist_subway_mean,
        "dist_bus_mean": dist_bus_mean,
        "transit_stop_count": transit_stop_count,
    })
    
    time.sleep(0.5)

df_transit = pd.DataFrame(records)
print(f"\nTransit data for {len(df_transit)} tracts")

In [ ]:
# ── Intersection density via OSMnx ────────────────────
import osmnx as ox

# Download walking network for the full study area bounding box
lat_min = df_tracts["tract_lat"].min() - 0.01
lat_max = df_tracts["tract_lat"].max() + 0.01
lon_min = df_tracts["tract_lon"].min() - 0.01
lon_max = df_tracts["tract_lon"].max() + 0.01

print(f"Downloading walking network for bbox: ({lat_min:.3f},{lon_min:.3f}) to ({lat_max:.3f},{lon_max:.3f})...")
G = ox.graph_from_bbox(bbox=(lat_min, lat_max, lon_min, lon_max), network_type="walk")
print(f"  Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Extract node coordinates
node_lats = np.array([G.nodes[n]["y"] for n in G.nodes])
node_lons = np.array([G.nodes[n]["x"] for n in G.nodes])

# For each tract centroid, count intersections within 500m
INTERSECTION_RADIUS = 500  # meters
AREA_KM2 = math.pi * (INTERSECTION_RADIUS / 1000) ** 2

int_densities = []
for _, row in df_tracts.iterrows():
    lat, lon = row["tract_lat"], row["tract_lon"]
    dists = np.array([haversine(lat, lon, nlat, nlon) for nlat, nlon in zip(node_lats, node_lons)])
    n_intersections = np.sum(dists <= INTERSECTION_RADIUS)
    int_densities.append(round(n_intersections / AREA_KM2, 1))

df_transit["intersection_density"] = int_densities
print(f"Intersection density computed for {len(df_transit)} tracts")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/05_accessibility.csv"
df_transit.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_transit)} rows x {df_transit.shape[1]} cols)")
print(df_transit.describe().round(1).to_string())